Prediction
>I think that SVM will make a better classifier than DT and kNN because it has performed as good or better than the other ones in the other HW2 cases. 

In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix, classification_report
from sklearn.impute import KNNImputer

df = pd.read_csv("../Data/phl_exoplanet_catalog.csv")
features = df[['S_MASS', 'P_PERIOD', 'P_DISTANCE']]
targets = np.logical_or(df.P_HABITABLE == 1, df.P_HABITABLE == 2).astype(int)

X_train, X_test, y_train, y_test = train_test_split(features, targets, test_size=0.2, stratify=targets)

imp = KNNImputer()
X_train_filled = imp.fit_transform(X_train)
X_test_filled = imp.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_filled)
X_test_scaled = scaler.transform(X_test_filled)

svm = SVC(kernel="linear", class_weight="balanced")
svm.fit(X_train_scaled, y_train)

y_pred = svm.predict(X_test_scaled)

print("test accuracy:", accuracy_score(y_test, y_pred))
print("test recall:", recall_score(y_test, y_pred))
print("confusion matrix:\n", confusion_matrix(y_test, y_pred))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(svm, X_train_scaled, y_train, cv=cv, scoring='recall')
print("CV recall mean:", cv_scores.mean())
print("CV recall std:", cv_scores.std())


param_grid = {
    'kernel': ['linear', 'rbf'],
    'C': [0.1, 1, 10, 100],
    'gamma': [0.01, 0.1, 1, 10]
}

grid_svm = GridSearchCV(
    SVC(class_weight='balanced', random_state=42),
    param_grid,
    cv=cv,
    scoring='recall',
)

grid_svm.fit(X_train_scaled, y_train)

print("best parameters:", grid_svm.best_params_)
print("best CV recall:", grid_svm.best_score_)

best_svm = grid_svm.best_estimator_
y_pred_best = best_svm.predict(X_test_scaled)

print("best SVM accuracy:", accuracy_score(y_test, y_pred_best))
print("best SVM recall:", recall_score(y_test, y_pred_best))
print("confusion matrix:\n", confusion_matrix(y_test, y_pred_best))


test accuracy: 0.8358024691358025
test recall: 0.8181818181818182
confusion matrix:
 [[668 131]
 [  2   9]]
CV recall mean: 0.7944444444444445
CV recall std: 0.13099806802835104
best parameters: {'C': 100, 'gamma': 10, 'kernel': 'rbf'}
best CV recall: 0.8666666666666666
best SVM accuracy: 0.8197530864197531
best SVM recall: 1.0
confusion matrix:
 [[653 146]
 [  0  11]]


The test accuracy went down slightly with the hyperparameters but the recall went up significantly so I think it is a better model. The best parameters were 'C': 10, 'gamma': 10, 'kernel': 'rbf'. CV recall std gives the generalization uncertainty. It is 0.08. 